In [ ]:
!pip install -q crewai crewai-tools
!pip install -q ddgs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 46.3 MB/s eta 0:00:00


In [ ]:
import os
from google.colab import userdata
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY2")

In [ ]:
from crewai import LLM

llm = LLM(
    model = "gemini/gemini-flash-latest",
    temperature = 0.2
)

In [ ]:
from crewai.tools import tool
from ddgs import DDGS

@tool("DuckDuckGo Search")
def search_tool(query: str) -> str:
  """Search the web using DuckDuckGO. Input is a search query.
  Returns the top results with title, link and short snippet."""
  try:
    results = DDGS().text(query, max_results = 5)
    return "\n\n".join(
        f"Title: {r['title']}\nLink: {r['href']}\nSnippet: {r['body']}"
        for r in results
    )
  except Exception as e:
    return f"Search failed: {e}"

In [ ]:
from crewai import Agent
from crewai_tools import FileWriterTool

file_writer = FileWriterTool()

researcher = Agent(
    role = "Senior Reasearch Analyst",
    goal = "Find accurate, current facts, statistics and real examples about {topic}",
    backstory = "You are a careful researcher. You search multiple sources and "
              "always note where each fact came from.",
    tools = [search_tool],
    llm = llm,
    verbose = True,
    allow_delegation = False,
)

writer = Agent(
    role = "Blog Writer",
    goal = "Turn research notes into an engaging 600-word blog post about{topic}",
    backstory = "You write clear, friendly blog posts for a general audience. "
              "You only use facts given to you in the research notes.",
    llm = llm,
    verbose = True,
    allow_delegation = False,
)

editor = Agent(
    role = "Editor",
    goal = "Polish the blog post about {topic} for tone, clarity and factual accuracy, then save it",
    backstory = "You are a strict editor. You fix awkward sentences, remove any claim "
              "not supported by the research, and keep the post around 600 words.",
    tools = [file_writer],
    llm = llm,
    verbose = True,
    allow_delegation = False,
)

In [ ]:
from crewai import Task

research_task = Task(
    description = "Research the topic: {topic}. Use the search tool to find recent facts, "
                "statistics and real-world examples. Collect at least 5 key points, "
                "each with its source link.",
    expected_output= "A bullet list of at least 5 key findings about {topic}, "
                    "each with a source link.",
    agent = researcher,
)

write_task = Task(
    description = "Edit the blog post for tone, clarity and factual accuracy. Remove any "
                "claim not supported by the research notes. Then save the final post "
                "using the file writer tool with the filename 'blog_post.md'.",
    expected_output = "A 600-word blog post in Markdown with a title, subheadings"
                      " and a conclusion.",
    agent = writer,
    context=[research_task],
)

edit_task = Task(
    description="Edit the blog post for tone, clarity and factual accuracy. Remove any "
                "claim not supported by the research notes. Then save the final post "
                "using the file writer tool with the filename 'blog_post.md'.",
expected_output="A 600-word blog post in Markdown with a title, subheadings"
                " and a conclusion.",
    agent=editor,
    context=[research_task, write_task],
)

In [ ]:
from crewai import Crew, Process

blog_crew = Crew(
    agents = [researcher, writer, editor],
    tasks = [research_task, write_task, edit_task],
    process = Process.sequential,
    verbose = True,
)

result = await blog_crew.kickoff_async(inputs = {"topic": "AI in Farming"})
print(result)

╭───────────────────────── 🚀 Crew Execution Started ──────────────────────────╮
│                                                                              │
│  Crew Execution Started                                                      │
│  Name: crew                                                                  │
│  ID: 48477901-ecc0-4c30-96ef-80a53edd9f62                                    │
│                                                                              │
│                                                                              │
╰──────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────── 📋 Task Started ───────────────────────────────╮
│                                                                              │
│  Task Started                                                                │
│  Name: Research the topic: AI in Farming. Use the search tool to find        │
│  recent facts, statistics a

/tmp/ipykernel_1813/1932896031.py:10: RuntimeWarning: coroutine 'Crew.kickoff_async' was never awaited
  result = await blog_crew.kickoff_async(inputs = {"topic": "AI in Farming"})
